# 03 Gold analysis

Build the four denormalized gold tables and read them as a template for
BI or SQL consumers.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ETL_DB_BACKEND"] = "sqlite"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.settings import get_settings

settings = get_settings()
print("backend:", settings.backend)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

## Build gold (full refresh) from silver

In [ ]:
from violence_etl.db import make_engine
from violence_etl.gold import build_gold

engine = make_engine(settings=settings)
summary = build_gold(engine, export_csv=True, export_dir="data/gold")
summary

## a. Top 5 departments by total crimes

`rank` 1..5, ordered by `total_crimes DESC`, tie broken by `departamento ASC`.

In [ ]:
pd.read_sql("select * from gold_top5_departments order by rank", engine)

## b. Total crimes by department (all departments)

In [ ]:
pd.read_sql(
    "select * from gold_crimes_by_department order by total_crimes desc",
    engine,
)

## c. Total crimes by year, quarter, month and weekday

Grain: `year x quarter x month x weekday_num` (unique).

In [ ]:
calendar = pd.read_sql(
    "select * from gold_crimes_by_calendar "
    "where year = (select max(year) from gold_crimes_by_calendar) "
    "order by quarter, month, weekday_num",
    engine,
)
calendar

In [ ]:
# Aggregate a compact view: crimes by weekday across the whole history.
pd.read_sql(
    "select weekday_num, weekday_name, sum(total_crimes) as crimes "
    "from gold_crimes_by_calendar group by weekday_num, weekday_name "
    "order by weekday_num",
    engine,
)

## d. Total crimes by electoral period

`period_order` uses the presentation order post, intermediate, pre, electoral.
`pct_of_total` is the share of the grand total, rounded to 2 decimals and sums to 100.

In [ ]:
periods = pd.read_sql(
    "select * from gold_crimes_by_electoral_period order by period_order", engine
)
periods

In [ ]:
print("sum pct_of_total:", periods["pct_of_total"].sum())
print("grand total     :", periods["total_crimes"].sum())

## Equivalent plain SQL

```sql
-- b. total crimes by department
SELECT cod_depto, departamento, SUM(cantidad) AS total_crimes
FROM silver_crimes GROUP BY cod_depto, departamento ORDER BY total_crimes DESC;

-- c. total crimes by year, quarter, month and weekday
SELECT year, quarter, month, weekday_num, weekday_name, SUM(cantidad) AS total_crimes
FROM silver_crimes
GROUP BY year, quarter, month, weekday_num, weekday_name;

-- d. total crimes by electoral period
SELECT electoral_period, SUM(cantidad) AS total_crimes
FROM silver_crimes GROUP BY electoral_period;
```